# Source Validation

Lightweight notebook for validating one landed official Yellow Taxi source month before or alongside the Bronze and Silver pipeline runs.

This notebook is intentionally small and validation-oriented. Reusable pipeline logic stays in `src/`.

## Environment Notes

- Run this notebook with the project-compatible environment from `projects/04-urban-mobility-analytics/requirements.txt`.
- The preferred path is Pandas plus `pyarrow`.
- If `pyarrow` is missing but `duckdb` is available in the active kernel, the notebook falls back to DuckDB for local Parquet inspection.
- If neither path is available, the bootstrap cell raises a clear error with the exact dependency expectation.
- Supported launch locations are the repository root, the project root, and this `notebooks/` directory.

In [7]:
from importlib import import_module
from pathlib import Path
import sys

PROJECT_SENTINEL = Path("src/config.py")
cwd = Path.cwd().resolve()
candidate_paths = [
    cwd,
    cwd.parent if cwd.name == "notebooks" else None,
    cwd / "projects" / "04-urban-mobility-analytics",
]

PROJECT_ROOT = None
for candidate in candidate_paths:
    if candidate is None:
        continue
    if (candidate / PROJECT_SENTINEL).exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    supported_locations = [
        "repository root",
        "projects/04-urban-mobility-analytics",
        "projects/04-urban-mobility-analytics/notebooks",
    ]
    raise RuntimeError(
        "Could not locate projects/04-urban-mobility-analytics from the current working directory. "
        f"Launch the notebook from one of these locations: {supported_locations}. Current cwd: {cwd}"
    )

PROJECT_SRC = PROJECT_ROOT / "src"
if str(PROJECT_SRC) not in sys.path:
    sys.path.insert(0, str(PROJECT_SRC))

pandas = import_module("pandas")
pd = pandas

pyarrow = None
pq = None
duckdb = None
PARQUET_BACKEND = None

try:
    pyarrow = import_module("pyarrow")
    pq = import_module("pyarrow.parquet")
    PARQUET_BACKEND = "pandas+pyarrow"
except ImportError:
    try:
        duckdb = import_module("duckdb")
        PARQUET_BACKEND = "duckdb"
    except ImportError as exc:
        raise RuntimeError(
            "This notebook requires a project-compatible kernel. Use Pandas with pyarrow, or install duckdb as a fallback Parquet reader. "
            "Activate the environment built from projects/04-urban-mobility-analytics/requirements.txt and reconnect the notebook kernel."
        ) from exc

if duckdb is None:
    duckdb = import_module("duckdb")

def preview_raw_file(raw_path: Path, columns: list[str], limit: int = 10):
    if PARQUET_BACKEND == "pandas+pyarrow":
        return pd.read_parquet(raw_path, columns=columns).head(limit)

    relation = duckdb.execute(
        f"SELECT {', '.join(columns)} FROM read_parquet(?) LIMIT {limit}",
        [str(raw_path)],
    )
    return relation.fetchdf()

def build_schema_summary(raw_path: Path, selected_columns: list[str]) -> dict[str, object]:
    if PARQUET_BACKEND == "pandas+pyarrow":
        parquet_file = pq.ParquetFile(raw_path)
        return {
            "row_count": int(parquet_file.metadata.num_rows),
            "column_count": len(parquet_file.schema_arrow.names),
            "first_10_columns": parquet_file.schema_arrow.names[:10],
            "selected_columns": selected_columns,
        }

    schema_rows = duckdb.execute(
        "DESCRIBE SELECT * FROM read_parquet(?)",
        [str(raw_path)],
    ).fetchall()
    return {
        "row_count": int(duckdb.execute("SELECT COUNT(*) FROM read_parquet(?)", [str(raw_path)]).fetchone()[0]),
        "column_count": len(schema_rows),
        "first_10_columns": [row[0] for row in schema_rows[:10]],
        "selected_columns": selected_columns,
    }

print(f"Resolved project root: {PROJECT_ROOT}")
print(f"Python executable: {sys.executable}")
print(f"Parquet backend: {PARQUET_BACKEND}")

Resolved project root: /home/willian/PhaifferTech/repos/data-engineering-portfolio/projects/04-urban-mobility-analytics
Python executable: /home/willian/PhaifferTech/repos/data-engineering-portfolio/.venv/bin/python
Parquet backend: duckdb


## Source Month Selection

In [8]:
from config import MonthPartition, build_bronze_raw_file_path

SOURCE_MONTH = MonthPartition.from_string("2024-01")
raw_path = build_bronze_raw_file_path(SOURCE_MONTH)

if not raw_path.exists():
    raise FileNotFoundError(
        "The selected raw source file does not exist yet. Run the ingestion job first, for example: "
        "python projects/04-urban-mobility-analytics/src/jobs/run_ingestion.py --start-month 2024-01 --end-month 2024-01"
    )

raw_path

PosixPath('/home/willian/PhaifferTech/repos/data-engineering-portfolio/projects/04-urban-mobility-analytics/data/bronze/raw/yellow_taxi/year=2024/month=01/source.parquet')

## Sample Schema Preview

In [9]:
sample_columns = [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "payment_type",
    "fare_amount",
    "total_amount",
]

sample_preview = preview_raw_file(raw_path, sample_columns, limit=10)
sample_preview

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,payment_type,fare_amount,total_amount
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,2,17.7,22.70
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.80,1,10.0,18.75
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.70,1,23.3,31.30
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.40,1,10.0,17.00
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.80,1,7.9,16.10
5,1,2024-01-01 00:54:08,2024-01-01 01:26:31,1,4.70,1,29.6,41.50
6,2,2024-01-01 00:49:44,2024-01-01 01:15:47,2,10.82,1,45.7,64.95
7,1,2024-01-01 00:30:40,2024-01-01 00:58:40,0,3.00,2,25.4,30.40
8,2,2024-01-01 00:26:01,2024-01-01 00:54:12,1,5.44,2,31.0,36.00
9,2,2024-01-01 00:28:08,2024-01-01 00:29:16,1,0.04,2,3.0,8.00


## File And Schema Summary

In [10]:
schema_summary = {
    "source_month": SOURCE_MONTH.month_id,
    **build_schema_summary(raw_path, sample_columns),
}
schema_summary

{'source_month': '2024-01',
 'row_count': 2964624,
 'column_count': 21,
 'first_10_columns': ['VendorID',
  'tpep_pickup_datetime',
  'tpep_dropoff_datetime',
  'passenger_count',
  'trip_distance',
  'RatecodeID',
  'store_and_fwd_flag',
  'PULocationID',
  'DOLocationID',
  'payment_type'],
 'selected_columns': ['VendorID',
  'tpep_pickup_datetime',
  'tpep_dropoff_datetime',
  'passenger_count',
  'trip_distance',
  'payment_type',
  'fare_amount',
  'total_amount']}

## Timestamp Bounds And Null Overview

In [11]:
validation_query = """
SELECT
    COUNT(*) AS row_count,
    MIN(tpep_pickup_datetime) AS pickup_min,
    MAX(tpep_pickup_datetime) AS pickup_max,
    MIN(tpep_dropoff_datetime) AS dropoff_min,
    MAX(tpep_dropoff_datetime) AS dropoff_max,
    SUM(CASE WHEN VendorID IS NULL THEN 1 ELSE 0 END) AS vendor_id_null_count,
    SUM(CASE WHEN passenger_count IS NULL THEN 1 ELSE 0 END) AS passenger_count_null_count,
    SUM(CASE WHEN trip_distance IS NULL THEN 1 ELSE 0 END) AS trip_distance_null_count,
    SUM(CASE WHEN payment_type IS NULL THEN 1 ELSE 0 END) AS payment_type_null_count,
    SUM(CASE WHEN fare_amount IS NULL THEN 1 ELSE 0 END) AS fare_amount_null_count,
    SUM(CASE WHEN total_amount IS NULL THEN 1 ELSE 0 END) AS total_amount_null_count
FROM read_parquet(?)
"""

validation_summary = duckdb.execute(validation_query, [str(raw_path)]).fetchdf()
validation_summary

,row_count,pickup_min,pickup_max,dropoff_min,dropoff_max,vendor_id_null_count,passenger_count_null_count,trip_distance_null_count,payment_type_null_count,fare_amount_null_count,total_amount_null_count
0,2964624,2002-12-31 22:59:39,2024-02-01 00:01:15,2002-12-31 23:05:41,2024-02-02 13:56:52,0.0,140162.0,0.0,0.0,0.0,0.0


## Source-Month Spillover Check

The official TLC files are planned and tracked by source month. That does not guarantee every record inside the file has a pickup timestamp inside that same calendar month.

Project 04 keeps that behavior visible:

- ingestion and state tracking stay aligned with the official source month;
- Silver and Gold partition by resolved pickup year and month;
- small spillover partitions are recorded in metadata instead of being silently filtered.

In [12]:
spillover_query = """
SELECT
    COUNT(*) FILTER (
        WHERE date_trunc('month', tpep_pickup_datetime) != date_trunc('month', CAST(? AS TIMESTAMP))
    ) AS pickup_outside_source_month_count,
    MIN(tpep_pickup_datetime) FILTER (
        WHERE date_trunc('month', tpep_pickup_datetime) != date_trunc('month', CAST(? AS TIMESTAMP))
    ) AS earliest_spillover_pickup,
    MAX(tpep_pickup_datetime) FILTER (
        WHERE date_trunc('month', tpep_pickup_datetime) != date_trunc('month', CAST(? AS TIMESTAMP))
    ) AS latest_spillover_pickup
FROM read_parquet(?)
"""

source_month_start = f"{SOURCE_MONTH.year:04d}-{SOURCE_MONTH.month:02d}-01 00:00:00"
spillover_summary = duckdb.execute(
    spillover_query,
    [source_month_start, source_month_start, source_month_start, str(raw_path)],
).fetchdf()
spillover_summary

,pickup_outside_source_month_count,earliest_spillover_pickup,latest_spillover_pickup
0,18,2002-12-31 22:59:39,2024-02-01 00:01:15
